# Ethiopia Food Prices — 09 ML FORECASTING

Build and compare machine-learning forecasting models for monthly price prediction.


In [2]:
# Use the cleaned raw dataset for the Addis Ababa maize story.
# This keeps the ML forecast notebook aligned with the project's time-series analysis,
# which selected the same representative market/commodity and pricing configuration.
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import joblib

root = Path('../')
df = pd.read_csv(root / 'data' / 'eth_food_prices_clean.csv', parse_dates=['date'])
df = df.sort_values(['market', 'commodity', 'date']).reset_index(drop=True)
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns.')


Loaded 62,875 rows x 21 columns.


In [3]:
# 1. Select the same representative series used in the time-series notebook.
# We keep Addis Ababa, Maize (white), Wholesale, and the 100 KG unit.
# This makes the forecasting notebook consistent with the overall project narrative.
focus_market = 'Addis Ababa'
focus_commodity = 'Maize (white)'
focus_unit = '100 KG'
focus_pricetype = 'Wholesale'

series = df[
    (df['market'] == focus_market)
    & (df['commodity'] == focus_commodity)
    & (df['pricetype'] == focus_pricetype)
    & (df['unit'] == focus_unit)
].copy()

series = series.sort_values('date').reset_index(drop=True)
series['date'] = pd.to_datetime(series['date'])

# Engineering the same lag and rolling features used in the feature notebook.
series['year'] = series['date'].dt.year
series['month_of_year'] = series['date'].dt.month
series['lag_1'] = series['usdprice'].shift(1)
series['lag_3'] = series['usdprice'].shift(3)
series['lag_6'] = series['usdprice'].shift(6)

for window in [3, 6, 12]:
    series[f'rolling_mean_{window}'] = series['usdprice'].shift(1).rolling(window=window, min_periods=1).mean()
    series[f'rolling_std_{window}'] = series['usdprice'].shift(1).rolling(window=window, min_periods=2).std()

series['pct_change_1m'] = series['usdprice'].pct_change().shift(1) * 100
series['price_to_mean_ratio'] = series['usdprice'] / series['usdprice'].mean()
series = series.dropna(subset=['lag_1', 'lag_3', 'lag_6']).reset_index(drop=True)

print(f'Rows for {focus_market} / {focus_commodity} / {focus_unit}: {len(series)}')
series[['date', 'usdprice', 'lag_1', 'rolling_mean_3', 'rolling_std_3']].head()


Rows for Addis Ababa / Maize (white) / 100 KG: 252


,date,usdprice,lag_1,rolling_mean_3,rolling_std_3
0,2000-07-15,15.49,15.84,15.966667,0.434089
1,2000-08-15,15.45,15.49,15.646667,0.177858
2,2000-09-15,14.16,15.45,15.593333,0.214554
3,2000-10-15,12.15,14.16,15.033333,0.756593
4,2000-11-15,10.81,12.15,13.920000,1.663039


In [4]:
# Build the model matrix from the same representative Addis Ababa maize series.
# We keep the time ordering intact to respect real forecasting behavior.
target_col = 'usdprice'
feature_cols = [
    'year', 'month_of_year', 'lag_1', 'lag_3', 'lag_6',
    'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12',
    'rolling_std_3', 'rolling_std_6', 'rolling_std_12',
    'pct_change_1m', 'price_to_mean_ratio'
]
X = series[feature_cols]
y = series[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)
print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')


Train size: 201 | Test size: 51


In [5]:
# Compare several regression models on the same series.
# LinearRegression is a strong baseline; RandomForest captures nonlinear relationships;
# XGBoost is a powerful boosted tree model and is well suited to tabular series features.
models = {
    'linear_regression': LinearRegression(),
    'random_forest': RandomForestRegressor(n_estimators=300, random_state=42),
    'xgboost': XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        objective='reg:squarederror'
    )
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mape = mean_absolute_percentage_error(y_test, pred) * 100
    results.append({'model': name, 'mae': mae, 'rmse': rmse, 'mape': mape})
    print(f'{name.upper()} -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, MAPE: {mape:.2f}%')

metrics_df = pd.DataFrame(results)
metrics_df = metrics_df.sort_values('rmse').reset_index(drop=True)
metrics_df.to_csv(root / 'models' / 'forecast_metrics.csv', index=False)
print('\nBest forecast model on RMSE:')
print(metrics_df.head())


LINEAR_REGRESSION -> MAE: 0.00, RMSE: 0.00, MAPE: 0.00%
RANDOM_FOREST -> MAE: 0.32, RMSE: 0.64, MAPE: 0.83%
XGBOOST -> MAE: 0.70, RMSE: 1.74, MAPE: 1.71%

Best forecast model on RMSE:
               model           mae          rmse          mape
0  linear_regression  5.140986e-14  5.827945e-14  1.619298e-13
1      random_forest  3.198797e-01  6.380689e-01  8.286564e-01
2            xgboost  6.989922e-01  1.737096e+00  1.710648e+00


In [6]:
# Save the best-performing model for future forecasting and reuse.
best_model_name = metrics_df.iloc[0]['model']
best_model = models[best_model_name]
joblib.dump(best_model, root / 'models' / 'maize_forecast_model.joblib')
print(f'BEST MODEL SAVED: {best_model_name}')


BEST MODEL SAVED: linear_regression
